# PathFinder: CSC 603 Capstone

This notebook **runs** the team's code. The code itself lives in the GitHub repo.

- Run the cells from top to bottom.
- Do not write project code in this notebook. Put it in your own file in the repo, then re-run section 1 to pull the latest version.

| Section | Owner | Files in the repo |
|---|---|---|
| 2. Data | Litton | `data/courses.json` |
| 3. Planner and validator | Galo | `pathfinder/validator.py`, `pathfinder/planner.py` |
| 4. AI | Delger | `pathfinder/skills.py`, `matching.py`, `explain.py`, `baseline.py`, `app.py` |
| 5. Testing | Everyone | `tests/`, `eval/run_eval.py` |

## 1. Project setup

Clones the repo (or pulls the latest changes if it is already here) and installs the packages.
Re-run this cell whenever someone pushes new code.

In [ ]:
import os, sys

REPO = "https://github.com/GaloReyes00/pathfinder.git"
DEST = "/content/pathfinder"

if os.path.isdir(os.path.join(DEST, ".git")):
    !git -C {DEST} pull -q
else:
    !git clone -q {REPO} {DEST}

assert os.path.isdir(DEST), "Clone failed. Check that the repo is pushed to GitHub and set to public."
os.chdir(DEST)
%pip install -q -r requirements.txt
if DEST not in sys.path:
    sys.path.insert(0, DEST)
print("Setup done. Working folder:", os.getcwd())

In [ ]:
import importlib

def show_status(checks):
    """Print whether each expected function exists yet. checks = [(module_name, function_name), ...]"""
    for module_name, function_name in checks:
        module = importlib.reload(importlib.import_module(module_name))
        state = "ready" if hasattr(module, function_name) else "not written yet"
        print(f"{module_name}.{function_name}: {state}")

## 2. Data (Litton)

Loads the course dataset from `data/courses.json`.

In [ ]:
import json, pathlib

courses_path = pathlib.Path("data/courses.json")
raw = courses_path.read_text().strip()
courses = json.loads(raw) if raw else []
print(f"{len(courses)} courses loaded from {courses_path}")
for course in courses[:5]:
    print(" ", course["id"], "-", course.get("title", ""))

## 3. Planner and validator (Galo)

`validate_plan(plan, profile)` returns a list of rule violations (empty list means the plan is valid).
`build_plan(profile, relevance)` returns a semester-by-semester plan.

In [ ]:
show_status([
    ("pathfinder.validator", "validate_plan"),
    ("pathfinder.planner", "build_plan"),
])

**Demo.** The validator checks one valid plan and one broken plan. It uses the made-up `TEST` courses in
`tests/sample_courses.json` until the real dataset is in `data/courses.json`.

In [ ]:
from pathfinder.validator import load_courses, validate_plan

sample_courses = load_courses("tests/sample_courses.json")
profile = {"completed": ["TEST 101"], "max_units": 7, "semesters": 3}

plans = {
    "Valid plan": [["TEST 201", "TEST 110"], ["TEST 301", "TEST 350"], ["TEST 400"]],
    "Broken plan": [["TEST 301", "TEST 999"], ["TEST 201", "TEST 101", "TEST 350"]],
}
for name, plan in plans.items():
    problems = validate_plan(plan, profile, courses=sample_courses, requirements={})
    print(f"{name}: {len(problems)} violation(s)")
    for problem in problems:
        print("  -", problem)

**Real SFSU data.** The same check on the dataset in `data/courses.json`, for a first-year student with nothing
completed. Degree requirements are left out here because the plans only cover the first two semesters.

In [ ]:
if courses:
    profile = {"completed": [], "max_units": 15, "semesters": 8}
    plans = {
        "Valid start": [["MATH 226", "CSC 101"], ["MATH 227", "CSC 215", "PHYS 220", "PHYS 222"]],
        "Broken start": [["CSC 220", "PHYS 220"], ["CSC 230", "CSC 999"]],
    }
    for name, plan in plans.items():
        problems = validate_plan(plan, profile, courses=courses, requirements={})
        print(f"{name}: {len(problems)} violation(s)")
        for problem in problems:
            print("  -", problem)
else:
    print("data/courses.json is empty, so the real-data demo is skipped.")

## 4. AI (Delger)

Skill extraction from job postings, course matching, explanations, and the LLM-only baseline.
Keep API keys in Colab Secrets (the key icon on the left). Never put a key in the repo.

In [ ]:
show_status([
    ("pathfinder.skills", "extract_skills"),
    ("pathfinder.matching", "score_courses"),
    ("pathfinder.explain", "explain_plan"),
    ("pathfinder.baseline", "baseline_plan"),
])

## 5. Testing and evaluation (everyone)

Runs the automated tests. Evaluation runs (30 profiles, PathFinder vs. the baseline) will go here too.

In [ ]:
!python -m pytest -q tests